# 01. EDA: Data Audit and Exploratory Dropout Analysis

This notebook is the main workspace for the upgraded dropout determinants research.

Core modules:
1. Data inspection and descriptive dropout signals
2. Staged early warning models
3. Nonlinear model benchmark
4. Variable-block ablation study
5. Dropout-specific SHAP interpretation
6. Student-risk similarity graph and risk clustering
7. Course-risk graph

All generated tables and plots should be saved under `results/`.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

DATA_DIR = Path("data")
RESULTS_DIR = Path("results")
RESULTS_DIR.mkdir(exist_ok=True)

DATA_PATH = DATA_DIR / "data.csv"
df = pd.read_csv(DATA_PATH, sep=";")
df.columns = df.columns.str.strip()

df.shape

(4424, 37)

In [2]:
df.head()

,Marital status,Application mode,Application order,Course,Daytime/evening attendance\t,Previous qualification,Previous qualification (grade),Nacionality,Mother's qualification,Father's qualification,...,Curricular units 2nd sem (credited),Curricular units 2nd sem (enrolled),Curricular units 2nd sem (evaluations),Curricular units 2nd sem (approved),Curricular units 2nd sem (grade),Curricular units 2nd sem (without evaluations),Unemployment rate,Inflation rate,GDP,Target
0,1,17,5,171,1,1,122.0,1,19,12,...,0,0,0,0,0.000000,0,10.8,1.4,1.74,Dropout
1,1,15,1,9254,1,1,160.0,1,1,3,...,0,6,6,6,13.666667,0,13.9,-0.3,0.79,Graduate
2,1,1,5,9070,1,1,122.0,1,37,37,...,0,6,0,0,0.000000,0,10.8,1.4,1.74,Dropout
3,1,17,2,9773,1,1,122.0,1,38,37,...,0,6,10,5,12.400000,0,9.4,-0.8,-3.12,Graduate
4,2,39,1,8014,0,1,100.0,1,37,38,...,0,6,6,6,13.000000,0,13.9,-0.3,0.79,Graduate


## 1. Basic Data Audit

In [3]:
print("Rows, columns:", df.shape)
print("Missing values:", int(df.isna().sum().sum()))
display(df["Target"].value_counts())
display(df.describe(include="all").T)

Rows, columns: (4424, 37)
Missing values: 0


Target
Graduate    2209
Dropout     1421
Enrolled     794
Name: count, dtype: int64

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
Marital status,4424.0,NaN,NaN,NaN,1.178571,0.605747,1.0,1.0,1.0,1.0,6.0
Application mode,4424.0,NaN,NaN,NaN,18.669078,17.484682,1.0,1.0,17.0,39.0,57.0
Application order,4424.0,NaN,NaN,NaN,1.727848,1.313793,0.0,1.0,1.0,2.0,9.0
Course,4424.0,NaN,NaN,NaN,8856.642631,2063.566416,33.0,9085.0,9238.0,9556.0,9991.0
Daytime/evening attendance\t,4424.0,NaN,NaN,NaN,0.890823,0.311897,0.0,1.0,1.0,1.0,1.0
Previous qualification,4424.0,NaN,NaN,NaN,4.577758,10.216592,1.0,1.0,1.0,1.0,43.0
Previous qualification (grade),4424.0,NaN,NaN,NaN,132.613314,13.188332,95.0,125.0,133.1,140.0,190.0
Nacionality,4424.0,NaN,NaN,NaN,1.873192,6.914514,1.0,1.0,1.0,1.0,109.0
Mother's qualification,4424.0,NaN,NaN,NaN,19.561935,15.603186,1.0,2.0,19.0,37.0,44.0
Father's qualification,4424.0,NaN,NaN,NaN,22.275316,15.343108,1.0,3.0,19.0,37.0,44.0


## 2. Target Setup

In [4]:
df = df.copy()
df["DropoutBinary"] = (df["Target"] == "Dropout").astype(int)

target_multiclass = "Target"
target_binary = "DropoutBinary"

df[["Target", "DropoutBinary"]].head()

,Target,DropoutBinary
0,Dropout,1
1,Graduate,0
2,Dropout,1
3,Graduate,0
4,Graduate,0


## 3. Feature Groups for Later Modeling

These groups will support staged early warning models and ablation analysis.

In [5]:
pre_enrollment_features = [
    "Marital status",
    "Application mode",
    "Application order",
    "Course",
    "Daytime/evening attendance",
    "Previous qualification",
    "Previous qualification (grade)",
    "Nacionality",
    "Mother's qualification",
    "Father's qualification",
    "Mother's occupation",
    "Father's occupation",
    "Admission grade",
    "Displaced",
    "Educational special needs",
    "Debtor",
    "Tuition fees up to date",
    "Gender",
    "Scholarship holder",
    "Age at enrollment",
    "International",
    "Unemployment rate",
    "Inflation rate",
    "GDP",
]

first_semester_features = [
    "Curricular units 1st sem (credited)",
    "Curricular units 1st sem (enrolled)",
    "Curricular units 1st sem (evaluations)",
    "Curricular units 1st sem (approved)",
    "Curricular units 1st sem (grade)",
    "Curricular units 1st sem (without evaluations)",
]

second_semester_features = [
    "Curricular units 2nd sem (credited)",
    "Curricular units 2nd sem (enrolled)",
    "Curricular units 2nd sem (evaluations)",
    "Curricular units 2nd sem (approved)",
    "Curricular units 2nd sem (grade)",
    "Curricular units 2nd sem (without evaluations)",
]

financial_demographic_features = [
    "Age at enrollment",
    "Gender",
    "Marital status",
    "Displaced",
    "Educational special needs",
    "Debtor",
    "Tuition fees up to date",
    "Scholarship holder",
    "International",
    "Nacionality",
]

parental_features = [
    "Mother's qualification",
    "Father's qualification",
    "Mother's occupation",
    "Father's occupation",
]

macro_features = ["Unemployment rate", "Inflation rate", "GDP"]
course_application_features = ["Course", "Application mode", "Application order"]
academic_features = first_semester_features + second_semester_features + [
    "Previous qualification (grade)",
    "Admission grade",
]

stage_feature_sets = {
    "A_pre_enrollment": pre_enrollment_features,
    "B_first_semester": pre_enrollment_features + first_semester_features,
    "C_second_semester": pre_enrollment_features + first_semester_features + second_semester_features,
}

ablation_feature_sets = {
    "academic_only": academic_features,
    "academic_plus_financial_demographic": sorted(set(academic_features + financial_demographic_features)),
    "academic_plus_parental": sorted(set(academic_features + parental_features)),
    "academic_plus_macro": sorted(set(academic_features + macro_features)),
    "academic_plus_course_application": sorted(set(academic_features + course_application_features)),
    "full": stage_feature_sets["C_second_semester"],
}

{k: len(v) for k, v in stage_feature_sets.items()}, {k: len(v) for k, v in ablation_feature_sets.items()}

({'A_pre_enrollment': 24, 'B_first_semester': 30, 'C_second_semester': 36},
 {'academic_only': 14,
  'academic_plus_financial_demographic': 24,
  'academic_plus_parental': 18,
  'academic_plus_macro': 17,
  'academic_plus_course_application': 17,
  'full': 36})

## 4. EDA Output Folder

All exploratory data analysis outputs are saved under `results/1.EDA/`.

In [ ]:
EDA_DIR = RESULTS_DIR / "1.EDA"
EDA_CSV_DIR = EDA_DIR / "csv"
EDA_PLOTS_DIR = EDA_DIR / "plots"
EDA_CSV_DIR.mkdir(parents=True, exist_ok=True)
EDA_PLOTS_DIR.mkdir(parents=True, exist_ok=True)

# Keep the original CSV unchanged, but strip whitespace from column names inside the notebook.
df.columns = df.columns.str.strip()
df["DropoutBinary"] = (df["Target"] == "Dropout").astype(int)

EDA_DIR

## 5. Data Cleaning and Quality Checks

In [ ]:
quality = pd.DataFrame({
    "column": df.columns,
    "dtype": [str(df[c].dtype) for c in df.columns],
    "missing_count": [int(df[c].isna().sum()) for c in df.columns],
    "missing_pct": [float(df[c].isna().mean()) for c in df.columns],
    "unique_count": [int(df[c].nunique(dropna=False)) for c in df.columns],
})

audit_summary = pd.DataFrame({
    "metric": ["rows", "columns", "missing_values_total", "duplicate_rows"],
    "value": [len(df), df.shape[1], int(df.isna().sum().sum()), int(df.duplicated().sum())],
})

quality.to_csv(EDA_CSV_DIR / "data_quality_summary.csv", index=False)
audit_summary.to_csv(EDA_CSV_DIR / "dataset_audit_summary.csv", index=False)

display(audit_summary)
display(quality.head(10))

## 6. Target Distribution Analysis

In [ ]:
target_distribution = df["Target"].value_counts().rename_axis("Target").reset_index(name="count")
target_distribution["percent"] = target_distribution["count"] / target_distribution["count"].sum()
target_distribution.to_csv(EDA_CSV_DIR / "target_distribution.csv", index=False)
target_distribution

## 7. Descriptive Dropout Signal Analysis

In [ ]:
categorical_signal_cols = [
    "Tuition fees up to date", "Debtor", "Scholarship holder", "Gender", "Displaced",
    "Daytime/evening attendance", "Marital status", "Course", "Application mode",
    "Previous qualification", "Nacionality", "Mother's qualification", "Father's qualification",
    "Mother's occupation", "Father's occupation",
]

categorical_tables = []
for col in categorical_signal_cols:
    if col not in df.columns:
        continue
    table = (
        df.groupby(col)
        .agg(n=("DropoutBinary", "size"), dropout_rate=("DropoutBinary", "mean"))
        .reset_index()
        .sort_values("dropout_rate", ascending=False)
    )
    table.insert(0, "variable", col)
    table = table.rename(columns={col: "category"})
    categorical_tables.append(table)

categorical_dropout_rates = pd.concat(categorical_tables, ignore_index=True)
categorical_dropout_rates.to_csv(EDA_CSV_DIR / "categorical_dropout_rates_all.csv", index=False)
categorical_dropout_rates.head(20)

## 8. Numeric Dropout Gradient Analysis

In [ ]:
quintile_cols = [
    "Age at enrollment", "Admission grade", "Previous qualification (grade)",
    "Curricular units 1st sem (approved)", "Curricular units 1st sem (grade)",
    "Curricular units 2nd sem (approved)", "Curricular units 2nd sem (grade)",
]

quintile_tables = []
for col in quintile_cols:
    bins = pd.qcut(df[col].rank(method="first"), 5, labels=False) + 1
    table = (
        df.assign(quintile=bins)
        .groupby("quintile")
        .agg(
            n=("DropoutBinary", "size"),
            min_value=(col, "min"),
            max_value=(col, "max"),
            dropout_rate=("DropoutBinary", "mean"),
        )
        .reset_index()
    )
    table.insert(0, "variable", col)
    quintile_tables.append(table)

numeric_quintile_dropout_rates = pd.concat(quintile_tables, ignore_index=True)
numeric_quintile_dropout_rates.to_csv(EDA_CSV_DIR / "numeric_quintile_dropout_rates.csv", index=False)
numeric_quintile_dropout_rates.head(15)

## 9. Academic Correlation Analysis

In [ ]:
academic_corr_cols = [c for c in df.columns if "Curricular units" in c] + [
    "Admission grade",
    "Previous qualification (grade)",
]

academic_corr = df[academic_corr_cols].corr()
academic_corr.to_csv(EDA_CSV_DIR / "academic_correlation_matrix.csv")

high_corr_pairs = []
for i, col_1 in enumerate(academic_corr.columns):
    for col_2 in academic_corr.columns[i + 1:]:
        corr_value = academic_corr.loc[col_1, col_2]
        if abs(corr_value) >= 0.65:
            high_corr_pairs.append({
                "feature_1": col_1,
                "feature_2": col_2,
                "correlation": corr_value,
                "abs_correlation": abs(corr_value),
            })

high_academic_correlations = pd.DataFrame(high_corr_pairs).sort_values("abs_correlation", ascending=False)
high_academic_correlations.to_csv(EDA_CSV_DIR / "high_academic_correlations.csv", index=False)
high_academic_correlations.head(20)

## 10. Next Modules

Implementation order:
1. Shared preprocessing pipeline
2. Staged model benchmark
3. Ablation study
4. Dropout-specific SHAP
5. Risk clustering plus SHAP
6. Student-risk similarity graph
7. Course-risk graph